# D3a 순차 데이터와 RNN·LSTM — 실습 (W7)

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.
> 이번 주는 무거운 데이터가 없습니다 — **손 계산과 작은 실험**으로 RNN의 원리를 완전히 익히는 주입니다.

**이 실습이 끝나면**
1. 시퀀스 규격 **(배치, 시간, 특징)** 을 다룬다
2. **손 계산을 재현**한다 — 누적기 [1,3,6] / 반쯤 잊기 [1,2.5,4.25]
3. **신호의 세 운명**(0.5/1.0/1.5) 곡선을 그린다 — 기울기 소실의 직관
4. `out[:, -1, :] == h` 와 **LSTM = RNN×4** 를 검산한다
5. **사인파 다음 값 예측**(회귀)으로 RNN을 실전 데뷔시킨다

**7단계 멘탈모델 초점:** 표현 + 모델

## Part A. 시퀀스 데이터 준비 — 새 규격 (배치, 시간, 특징)
이번 주 재료는 가장 깨끗한 시퀀스, 사인파입니다.

In [ ]:
import torch                                            # PyTorch
import torch.nn as nn                                   # 신경망 모듈
import matplotlib.pyplot as plt                         # 그래프
import math                                             # pi

torch.manual_seed(0)                                    # 재현성
series = torch.sin(torch.linspace(0, 6 * math.pi, 120)) # 사인파 3주기, 120점
print('series:', series.shape)                          # (120,) — 아직 그냥 벡터

x_demo = series[:20].view(1, 20, 1)                     # (배치1, 시간20, 특징1) — 새 규격!
print('시퀀스 규격:', x_demo.shape)                     # (1, 20, 1) = (B, T, F)
plt.figure(figsize=(7, 2.5))                            # 데이터를 눈으로(습관)
plt.plot(series)                                        # 사인파
plt.title('Our sequence: sine wave (120 steps)')        # 제목(영어)
plt.grid(True); plt.show()                              # 순서 속 패턴 = 정보

## Part B. 손 계산 재현 ⭐ — 세상에서 가장 작은 RNN
`h = Wx·x + Wh·h이전` (활성화 없음). 설명서 §4의 두 표를 코드로 재현합니다.

In [ ]:
def tiny_rnn(xs, Wx, Wh):                               # 은닉 1칸짜리 RNN
    h = 0.0                                             # h0 = 0
    hist = []                                           # 기억의 역사
    for x in xs:                                        # 시점 순서대로
        h = Wx * x + Wh * ___                           # ✍️ 빈칸: 직전 기억(은닉 상태)
        hist.append(round(h, 4))                        # 기록
    return hist

print('누적기   (Wx=1, Wh=1)  :', tiny_rnn([1., 2., 3.], 1.0, 1.0))   # [1, 3, 6] — 달리는 합계
print('반쯤 잊기(Wx=1, Wh=0.5):', tiny_rnn([1., 2., 3.], 1.0, 0.5))   # [1, 2.5, 4.25]
print('전부 잊기(Wx=1, Wh=0)  :', tiny_rnn([1., 2., 3.], 1.0, 0.0))   # [1, 2, 3] — 기억 없음=MLP와 같음

> **Wh = 기억 강도.** 1이면 전부 기억(누적기), 0.5면 반쯤, 0이면 기억 없음(사실상 순서를 안 보는 것). RNN 학습 = 이 가중치들을 데이터에서 찾기(D1b).

## Part C. 신호의 세 운명 — 반복 곱셈의 병
옛 기억은 t시점 뒤 Wh^t 배. 0.5/1.0/1.5의 운명을 그립니다 (D1b "학습률 세 운명"의 시간판).

In [ ]:
plt.figure(figsize=(6, 4))                              # 세 곡선
for w in [0.5, 1.0, 1.5]:
    signal = [w ** t for t in range(21)]                # t시점 뒤 남은 신호
    plt.semilogy(signal, 'o-', label=f'Wh={w}')         # 로그 축(폭발·소멸을 한 화면에)
plt.xlabel('time steps'); plt.ylabel('remaining signal (log)')  # 축(영어)
plt.title('Three fates of memory: vanish / hold / explode')     # 제목(영어)
plt.legend(); plt.grid(True); plt.show()                # 20시점: 1e-6 vs 1 vs 3325
print('0.5^20 =', round(0.5 ** 20, 8), '| 1.5^20 =', round(1.5 ** 20, 1))  # 수치 확인

> 역전파도 같은 길을 거꾸로 곱하며 돌아오므로(연쇄법칙 — D1b) **기울기도 똑같이** 소멸/폭발 — 20단어 문장 = 사실상 20층. 이것이 장기 의존성 문제, LSTM의 존재 이유입니다.

## Part D. nn.RNN 규격 — batch_first와 out vs h
시험 단골 두 가지를 코드로 확정합니다.

In [ ]:
rnn = nn.RNN(input_size=1, hidden_size=16, batch_first=___)  # ✍️ 빈칸: (배치,시간,특징) 규격 사용!
out, h = rnn(x_demo)                                    # 20시점 통과
print('out:', out.shape)                                # (1, 20, 16) — 모든 시점의 은닉
print('h  :', h.shape)                                  # (1, 1, 16) — 마지막 시점만
print('out의 마지막 시점 == h ?',
      torch.allclose(out[:, ___, :], h[0]))             # ✍️ 빈칸: 음수 인덱싱(뒤에서 첫 번째) → True

## Part E. LSTM = RNN × 4 — 게이트 검산
게이트 3개 + 후보 1개 = 가중치 4벌. 파라미터로 확인합니다.

In [ ]:
n_rnn = sum(p.numel() for p in nn.RNN(1, 16).parameters())   # RNN 파라미터(D1c numel)
lstm = nn.___(1, 16, batch_first=True)                  # ✍️ 빈칸: 게이트 달린 셀
n_lstm = sum(p.numel() for p in lstm.parameters())      # LSTM 파라미터
print('RNN :', n_rnn, '| LSTM:', n_lstm, '| 비율:', n_lstm / n_rnn)  # 304 / 1216 / 4.0 — 정확히 4배!

out2, (h2, c2) = lstm(x_demo)                           # 반환에 벨트(c)가 하나 더
print('h:', h2.shape, '| c(벨트):', c2.shape)           # 둘 다 (1,1,16)

## Part F. 첫 실전 — 사인파 다음 값 예측 (회귀) ⭐⭐
**윈도우 썰기**: [t..t+19] 20개를 보고 → t+20 값을 예측. **분할은 시간 순서로**(과거 학습·미래 시험 — M2).

In [ ]:
WINDOW = 20                                             # 윈도우 길이
X_list, y_list = [], []
for i in range(len(series) - WINDOW):                   # 한 칸씩 밀며 썰기
    X_list.append(series[i:i + WINDOW])                 # 입력: 20개
    y_list.append(series[i + ___])                      # ✍️ 빈칸: 바로 다음 값의 인덱스 오프셋
X = torch.stack(X_list).unsqueeze(-1)                   # (100, 20, 1) — (B,T,F) 규격
y = torch.stack(y_list).view(-1, 1)                     # (100, 1)

n_tr = 80                                               # ⚠️ 시간 순서 분할: 앞 80 학습
Xtr, ytr, Xte, yte = X[:n_tr], y[:n_tr], X[n_tr:], y[n_tr:]  # 뒤 20 시험(미래)
print('전체:', X.shape, '| train:', Xtr.shape[0], '| test(미래):', Xte.shape[0])

In [ ]:
class SineRNN(nn.Module):                               # RNN 회귀 모델
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(1, 16, batch_first=True)      # 기억 16칸
        self.fc = nn.Linear(16, 1)                      # 요약 → 다음 값 1개
    def forward(self, x):
        out, h = self.rnn(x)                            # 시퀀스 읽기
        return self.fc(out[:, -1, :])                   # 마지막 요약으로 예측(Part D!)

model = SineRNN()                                       # 생성
criterion = nn.___()                                    # ✍️ 빈칸: 회귀 손실(M4·D1b — CE 아님!)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # Adam
for epoch in range(200):                                # D1c 리듬 그대로(전체 배치)
    optimizer.zero_grad()                               # ② 초기화
    loss = criterion(model(Xtr), ytr)                   # ③④ 순전파+손실
    loss.backward()                                     # ⑤a 역전파
    optimizer.step()                                    # ⑤b 갱신
print('최종 train MSE:', round(loss.item(), 6))         # ~0에 근접

In [ ]:
model.eval()                                            # 평가 모드(D1c)
with torch.no_grad():                                   # 기록 끄기
    pred = model(Xte)                                   # 미래 20점 예측
test_mse = nn.functional.mse_loss(pred, yte).item()     # 시험 MSE
print('test(미래) MSE:', round(test_mse, 6))            # 작을수록 좋음

plt.figure(figsize=(7, 4))                              # 예측 vs 실제
plt.plot(yte.squeeze(), 'o-', label='actual (future)')  # 실제 미래
plt.plot(pred.squeeze(), 'x--', label='RNN prediction') # RNN 예측
plt.xlabel('future step'); plt.ylabel('value')          # 축(영어)
plt.title('Sine next-value prediction (test = unseen future)')  # 제목(영어)
plt.legend(); plt.grid(True); plt.show()                # 곡선이 겹치면 성공 — 위상(기억)을 읽은 것

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "Wx=2, Wh=1로 [1,2,3]을 넣으면 h가 어떻게 되는지 내가 계산할 테니 채점해 줘." (그리고 tiny_rnn으로 검증!)
- "0.9^50이 왜 장기 의존 실패를 뜻하는지 내 설명을 들어줘."
- "nn.LSTM(8, 32)의 파라미터 수를 내가 계산할 테니 검산해 줘." (공식: 4×(8×32+32×32+2×32))
- "시계열을 무작위로 나누면 안 되는 이유를 M2와 연결해 설명해 볼게."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. 손 계산(누적기 [1,3,6]·반쯤 잊기 [1,2.5,4.25])으로 "Wh=기억 강도"를 체득했다
2. 세 운명 곡선과 LSTM=RNN×4(304/1,216) 검산으로 병과 처방을 이해했다
3. 윈도우 썰기 + 시간 순서 분할로 사인파 미래 예측에 성공했다(D1c 루프 재사용)

**스스로 점검**
- [ ] (배치, 시간, 특징) 규격과 batch_first의 관계를 안다
- [ ] out[:, -1, :] == h[0]인 이유를 안다
- [ ] LSTM 파라미터가 4배인 이유(게이트 4벌)를 안다
- [ ] 시계열 분할을 시간 순서로 하는 이유를 M2로 설명할 수 있다

**🔹심화 (선택)**
- Part F의 RNN을 **LSTM으로 교체**해 보세요(반환이 `out, (h, c)`로 바뀌는 것 주의). 사인파처럼 짧은 의존에선 차이가 작음을 확인.
- 사인파에 잡음을 섞어(`series + 0.1*torch.randn(120)`) 재학습 — 예측이 얼마나 버티는지.
- WINDOW를 5로 줄이면? 40으로 늘리면? — 윈도우 길이와 예측 품질의 관계를 표로 정리.
- tiny_rnn에 tanh를 씌워(`math.tanh(...)`) 누적기(Wh=1)를 다시 돌려 보세요 — 값이 1 근처에서 포화되는 것 확인(§5의 이유).